# GVHMR 추론 — Kaggle 무료 GPU 버전

`motion-retarget-poc` 핵심 기술 1(영상 → 3D 모션 추정) 단계를 Colab 대신 **Kaggle 무료 GPU**로 돌리는 노트북이다.
Colab 무료 GPU 쿼터가 막혔을 때의 대안 트랙 — 코드 자체는 Colab 버전(`colab/gvhmr_inference.ipynb`)과 동일한 파이프라인이고, Kaggle 환경에 맞춰 셋업 방식만 바꿨다(`condacolab` 불필요 — Kaggle 이미지에는 `/opt/conda`가 이미 있다).

## 시작 전에 반드시 할 것 (사람이 직접, 최초 1회 + 매 세션)

1. **Kaggle 계정 + 휴대폰 인증** — GPU/인터넷 접근에는 전화번호 인증이 필수다 (account settings → Phone Verification).
2. 노트북 우측 패널 **Settings**:
   - **Accelerator**: `GPU T4 x2` (또는 `GPU P100`) 선택
   - **Internet**: `On` (기본 꺼져 있음 — 꺼진 채로 두면 아래 pip/git 명령이 전부 실패한다)
   - **Persistence**: `Files only` 권장 (세션 껐다 켜도 `/kaggle/working` 유지)
3. **SMPL/SMPLX 바디 모델 + 캐릭터 FBX를 Kaggle Dataset으로 업로드** (최초 1회):
   - 이미 갖고 계신 `SMPL_NEUTRAL.pkl`, `SMPLX_NEUTRAL.npz`, `Ch28_nonPBR.fbx` 3개 파일(Google Drive `SMPL/` 폴더에 보관 중인 것)을 로컬로 내려받는다.
   - Kaggle → Create → New Dataset → 위 3개 파일 업로드 → 데이터셋 이름 예: `gvhmr-body-models` (비공개로 유지).
   - 이 노트북 우측 **Add Input** → 방금 만든 데이터셋 추가 → `/kaggle/input/gvhmr-body-models/`에 자동으로 마운트된다.
   - 이후 노트북부터는 이 데이터셋만 다시 Add Input 하면 되고, 파일을 매번 새로 올릴 필요는 없다.

무료 쿼터: **주당 GPU 30시간**, 세션당 최대 12시간. Colab보다 쿼터가 넉넉하고 리셋 주기가 명확하다(주 단위 UTC 기준).

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. Python 3.10 conda 환경 구성

Kaggle 이미지에는 conda가 `/opt/conda`에 이미 설치돼 있어서 Colab처럼 `condacolab`을 따로 설치할 필요가 없다.

In [ ]:
!conda create -n gvhmr python=3.10 -y -q
PY = "/opt/conda/envs/gvhmr/bin/python"
PIP = "/opt/conda/envs/gvhmr/bin/pip"
print(PY, PIP)

## 2. GVHMR 클론 + 의존성 설치

In [ ]:
%cd /kaggle/working
!git clone -q https://github.com/zju3dv/GVHMR.git
%cd GVHMR
!/opt/conda/envs/gvhmr/bin/pip install -q numpy==1.23.5 setuptools==68.0.0 wheel
!/opt/conda/envs/gvhmr/bin/pip install -q --no-build-isolation chumpy
!/opt/conda/envs/gvhmr/bin/pip install -q -r requirements.txt
!/opt/conda/envs/gvhmr/bin/pip install -q -e .

In [ ]:
!/opt/conda/envs/gvhmr/bin/python -c "import torch, hmr4d; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available()); print('hmr4d OK')" 

## 3. 체크포인트 다운로드

Colab 때와 동일하게 공식 Google Drive 링크(쿼터 초과로 자주 막힘) 대신 HuggingFace 미러를 쓴다.

In [ ]:
!/opt/conda/envs/gvhmr/bin/pip install -q huggingface_hub
!/opt/conda/envs/gvhmr/bin/hf download camenduru/GVHMR \
  "gvhmr/gvhmr_siga24_release.ckpt" \
  "hmr2/epoch=10-step=25000.ckpt" \
  "vitpose/vitpose-h-multi-coco.pth" \
  "dpvo/dpvo.pth" \
  --local-dir /kaggle/working/GVHMR/inputs/checkpoints

In [ ]:
!mkdir -p /kaggle/working/GVHMR/inputs/checkpoints/yolo
!/opt/conda/envs/gvhmr/bin/python -c "from ultralytics import YOLO; YOLO('yolov8x.pt')"
!mv /kaggle/working/GVHMR/yolov8x.pt /kaggle/working/GVHMR/inputs/checkpoints/yolo/yolov8x.pt

## 4. SMPL / SMPL-X / 캐릭터 FBX — Dataset에서 복사

위에서 Add Input으로 붙인 `gvhmr-body-models` 데이터셋에서 그대로 복사한다. 데이터셋 이름이 다르면 아래 경로의 `gvhmr-body-models`를 실제 이름으로 바꿔야 한다.

In [ ]:
import os, shutil, glob
print(glob.glob("/kaggle/input/*/*"))

In [ ]:
import os, shutil
os.makedirs("/kaggle/working/GVHMR/inputs/checkpoints/body_models/smpl", exist_ok=True)
os.makedirs("/kaggle/working/GVHMR/inputs/checkpoints/body_models/smplx", exist_ok=True)
os.makedirs("/kaggle/working/GVHMR/blender/assets", exist_ok=True)

DATASET_DIR = "/kaggle/input/gvhmr-body-models"  # 데이터셋 이름 다르면 여기만 수정

shutil.copy(f"{DATASET_DIR}/SMPL_NEUTRAL.pkl",
            "/kaggle/working/GVHMR/inputs/checkpoints/body_models/smpl/SMPL_NEUTRAL.pkl")
shutil.copy(f"{DATASET_DIR}/SMPLX_NEUTRAL.npz",
            "/kaggle/working/GVHMR/inputs/checkpoints/body_models/smplx/SMPLX_NEUTRAL.npz")
shutil.copy(f"{DATASET_DIR}/Ch28_nonPBR.fbx",
            "/kaggle/working/GVHMR/blender/assets/mixamo_character.fbx")

for p in [
    "/kaggle/working/GVHMR/inputs/checkpoints/body_models/smpl/SMPL_NEUTRAL.pkl",
    "/kaggle/working/GVHMR/inputs/checkpoints/body_models/smplx/SMPLX_NEUTRAL.npz",
    "/kaggle/working/GVHMR/blender/assets/mixamo_character.fbx",
]:
    print(os.path.exists(p), os.path.getsize(p), p)

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/GVHMR")
from hmr4d.utils.body_model.smpl_lite import SmplLite
from hmr4d.utils.body_model.smplx_lite import SmplxLite
SmplLite(model_path="/kaggle/working/GVHMR/inputs/checkpoints/body_models/smpl", gender="neutral")
SmplxLite(model_path="/kaggle/working/GVHMR/inputs/checkpoints/body_models/smplx", gender="neutral")
print("SMPL / SMPL-X 둘 다 정상 로드됨")

## 5. 추론 실행

먼저 GVHMR 내장 테니스 샘플 영상으로 돌린다(우리가 이미 검증에 쓰던 영상과 동일).

In [ ]:
%cd /kaggle/working/GVHMR
!/opt/conda/envs/gvhmr/bin/python tools/demo/demo.py --video docs/example_video/tennis.mp4 -s

결과는 `outputs/demo/tennis/hmr4d_results.pt`에 생긴다. **다음에 뭘 하든 이 파일부터 Kaggle Dataset이나 Google Drive 등 영구 저장소에 복사해둘 것** — 세션이 끝나면 `/kaggle/working`도 언젠가 초기화된다.